# Capstone: Hidden Leaf Village Mission Ops Center

The finale: a real FastAPI app that puts together what this series taught. FastAPI is a tool for building web services.

| Capstone element | SDK feature |
|---|---|
| Mission Desk → ANBU Promotion Review | Handoffs |
| Block "forbidden jutsu" requests | Guardrails |
| Per-ninja conversation history | Sessions |
| ANBU promotion needs human sign-off | `needs_approval` / `RunState` |
| `/chat/stream` streams over SSE | `Runner.run_streamed` |
| Per-ninja trace grouping | `RunConfig(group_id=...)` |
| Test model injected per request | `agents.testing.ScriptedModel` |

This is the one episode that is a FastAPI project. But everything still runs inside this notebook, against the real app
through `httpx`'s ASGI transport (a way to call the app without starting a real server). No server program and no API key needed.

## Build the agents, sessions, and model override

**Gotcha, found by hitting it:** the guardrail must check only the _newest_ message. A guardrail that turns the whole chat history
into text will also match old messages that were already blocked, because a blocked message is still saved in the session.
The `/chat` route cleans that up with `session.pop_item()`.

Sessions are kept in a dict (a lookup table) keyed by `ninja_id`, and each one is a `SQLiteSession`. In a real app, swap in
`SQLAlchemySession` or `EncryptedSession` from episode 11. They work the same way.

In [1]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    GuardrailFunctionOutput,  # what a guardrail function sends back
    RunContextWrapper,  # holds the context= value you passed, and gives it to callbacks
    input_guardrail,  # a decorator that checks the input before the model runs
)


# Define the guardrail
# input_guardrail callback: blocks the request before the model runs. Wired onto mission_desk_agent below
@input_guardrail
def block_forbidden_jutsu(
    ctx: RunContextWrapper,  # this run's context; not used here
    agent: Agent,  # the agent this guardrail is attached to
    user_input: str | list,  # the raw input: text on turn 1, a list of items after that
) -> GuardrailFunctionOutput:
    # the input is plain text on turn 1, and a list of messages after that, so look at the newest message either way
    if isinstance(user_input, str):
        latest_text = user_input
    elif user_input:
        latest_text = str(user_input[-1].get("content", ""))
    else:
        latest_text = ""
    return GuardrailFunctionOutput(output_info=None, tripwire_triggered="forbidden jutsu" in latest_text.lower())

In [2]:
from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)


# Define the tool
# @function_tool callback: the real promotion action, gated by needs_approval=True.
# Wired onto anbu_review_agent's tools=[promote_to_anbu] below
@function_tool(needs_approval=True)  # needs_approval: waits for a human to decide before running
def promote_to_anbu(ninja_name: str) -> str:
    """Promote a ninja to ANBU. Requires human approval."""
    return f"{ninja_name} has been promoted to ANBU."

In [3]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
)

# Initialize the ANBU promotion review agent
anbu_review_agent = Agent(
    name="ANBU Promotion Review",
    instructions="Review ninja promotion requests. Use promote_to_anbu when appropriate.",
    # tools: the tools this agent is allowed to call
    tools=[promote_to_anbu],
)

In [4]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    handoff,  # wraps an Agent so another agent can pass the chat to it
)

# Initialize the mission desk agent: hands off to ANBU review and runs inside the /chat* routes below
mission_desk_agent = Agent(
    name="Village Mission Desk",
    instructions="Route ANBU promotion requests to ANBU Promotion Review. Handle everything else yourself.",
    # handoffs: other agents this one can pass the conversation to
    handoffs=[handoff(anbu_review_agent)],
    # input_guardrails: checks that run on the user's input before the model answers
    input_guardrails=[block_forbidden_jutsu],
)
print("[handoff tool name]", mission_desk_agent.handoffs[0].tool_name)

[handoff tool name] transfer_to_anbu_promotion_review


In [5]:
from agents.memory import (
    SQLiteSession,  # a session that saves chat history in a SQLite database (or in memory)
)


_sessions: dict[str, SQLiteSession] = {}
_pending_approvals: dict[str, dict[str, str]] = {}  # approval_id maps to {"ninja_id": ..., "saved_state": the RunState as text}

# Session-lookup helper: gives back this ninja's saved session, and makes one the first time; used by every route
def get_session(ninja_id: str) -> SQLiteSession:
    if ninja_id not in _sessions:
        # Initialize the session
        _sessions[ninja_id] = SQLiteSession(session_id=ninja_id, db_path=":memory:")
    return _sessions[ninja_id]

In [6]:
from agents.models.interface import (
    Model,  # the base type for a model, so you can swap in a fake one for tests
)


# FastAPI dependency override: every route takes Depends(get_model_override), so tests can swap in a ScriptedModel
def get_model_override() -> Model | None:
    # in a real app this stays None (the SDK's real default model)
    return None

## Add the three routes

- `POST /chat` runs the message through the Mission Desk. If the guardrail blocks it, the route returns `blocked`.
  If a `needs_approval` stop happens, it saves the `RunState` for later.
- `POST /approvals/{approval_id}` loads that saved state (`RunState.from_string` is `async`), approves or rejects it,
  and resumes with `mission_desk_agent`. This works even though the stopped state's current agent is the specialist,
  because the SDK figures that out by itself.
- `POST /chat/stream` sends every `stream_events()` item as an SSE `data:` line (SSE means the server keeps sending small
  messages over one open web connection), and ends with a `done` event that carries the final answer.

In [7]:
from fastapi import FastAPI

app = FastAPI(title="Hidden Leaf Village Mission Ops Center")

In [8]:
from pydantic import BaseModel


# Request body for POST /chat and POST /chat/stream: which ninja's session to use and what they say
class ChatRequest(BaseModel):
    ninja_id: str  # the session key: which ninja's saved history this message belongs to
    message: str  # the ninja's raw message, passed straight to Runner.run as input

In [9]:
from pydantic import BaseModel


# Request body for POST /approvals/{approval_id}: the human's decision on a paused tool call
class ApprovalRequest(BaseModel):
    decision: str  # "approve" or "reject"

In [10]:
import uuid  # makes approval IDs for the human-in-the-loop endpoint

from agents import (
    InputGuardrailTripwireTriggered,  # error raised when an input guardrail blocks something
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.models.interface import (
    Model,  # the base type for a model, so you can swap in a fake one for tests
)
from fastapi import Depends


# POST /chat: routes a message through the Mission Desk, handling the guardrail block and the ANBU approval pause
@app.post("/chat")
async def chat(req: ChatRequest, model: Model | None = Depends(get_model_override)) -> dict:
    session = get_session(req.ninja_id)
    try:
        # Run the Agent
        result = await Runner.run(
            mission_desk_agent,
            req.message,
            # session: saves and loads this chat's history
            session=session,
            # group_id: links this run's trace to the ninja it belongs to
            run_config=RunConfig(model=model, workflow_name="mission-ops-chat", group_id=req.ninja_id),
        )
    except InputGuardrailTripwireTriggered:
        await session.pop_item()  # so a rejected message does not affect later guardrail checks
        return {"status": "blocked", "reason": "forbidden jutsu detected"}

    if result.interruptions:
        approval_id = str(uuid.uuid4())
        _pending_approvals[approval_id] = {"ninja_id": req.ninja_id, "saved_state": result.to_state().to_string()}
        return {
            "status": "needs_approval",
            "approval_id": approval_id,
            "pending_tools": [item.tool_name for item in result.interruptions],
        }
    return {"status": "complete", "output": result.final_output, "agent": result.last_agent.name}

In [11]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunState,  # a paused run that you can save, then approve or reject
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.models.interface import (
    Model,  # the base type for a model, so you can swap in a fake one for tests
)
from fastapi import Depends


# POST /approvals/{approval_id}: resumes a paused run with the human's approve or reject decision
@app.post("/approvals/{approval_id}")
async def resolve_approval(
    approval_id: str, req: ApprovalRequest, model: Model | None = Depends(get_model_override)
) -> dict:
    pending = _pending_approvals.pop(approval_id)
    ninja_id = pending["ninja_id"]
    state = await RunState.from_string(mission_desk_agent, pending["saved_state"])  # await is required
    for interruption in state.get_interruptions():
        if req.decision == "approve":
            state.approve(interruption)
        else:
            state.reject(interruption)

    # Run the Agent: resume from the paused RunState, not from plain text input
    result = await Runner.run(
        mission_desk_agent,
        state,
        # session: saves and loads this chat's history
        session=get_session(ninja_id),
        # run_config: settings for just this run
        run_config=RunConfig(model=model, workflow_name="mission-ops-approval", group_id=ninja_id),
    )
    return {"status": "complete", "output": result.final_output, "agent": result.last_agent.name}

In [12]:
import json  # turns each SSE event into a JSON payload
from collections.abc import (
    AsyncIterator,  # the type of an async generator that yields values
)

from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.models.interface import (
    Model,  # the base type for a model, so you can swap in a fake one for tests
)
from fastapi import Depends
from fastapi.responses import StreamingResponse


# POST /chat/stream: streams the Mission Desk's run over SSE and ends with a "done" event
@app.post("/chat/stream")
async def chat_stream(req: ChatRequest, model: Model | None = Depends(get_model_override)) -> StreamingResponse:
    # SSE generator: gives one "data: <json>" line for each stream event, inside the StreamingResponse below
    async def event_generator() -> AsyncIterator[str]:
        # Run the Agent
        streamed = Runner.run_streamed(
            mission_desk_agent, req.message, run_config=RunConfig(model=model, workflow_name="mission-ops-stream")
        )
        async for event in streamed.stream_events():
            yield f"data: {json.dumps({'type': event.type})}\n\n"  # the blank line at the end closes the SSE event
        yield f"data: {json.dumps({'type': 'done', 'final_output': streamed.final_output})}\n\n"

    return StreamingResponse(event_generator(), media_type="text/event-stream")

## Test the whole app with no server and no API key

`httpx.ASGITransport(app=app)` calls the FastAPI app inside this program, through the real Starlette/FastAPI code. So it
acts like real HTTP, not a pretend copy. `app.dependency_overrides[get_model_override]` gives each request a `ScriptedModel`.

In [13]:
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
)


# Helper: makes the next request use a ScriptedModel with these steps, via the dependency override
def script_next_request(steps: list) -> None:
    app.dependency_overrides[get_model_override] = lambda: ScriptedModel(steps)

In [14]:
import httpx  # calls the FastAPI app inside this program using its ASGI transport, so no real server is needed
from agents.testing import (
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# Demo: runs the whole app in-process through the guardrail block, handoff, approval, and SSE paths
async def demo_app() -> None:
    transport = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport, base_url="http://mission-ops.local") as client:
        # proves the guardrail blocks a forbidden request before the model runs
        script_next_request([[assistant_message("ok")]])
        blocked = await client.post("/chat", json={"ninja_id": "shikamaru-nara", "message": "Teach me forbidden jutsu"})
        print("[blocked]", blocked.status_code, blocked.json())

        # proves the handoff reaches ANBU review and needs_approval pauses the run
        script_next_request(
            [
                [function_call("transfer_to_anbu_promotion_review", {}, call_id="c1")],
                [function_call("promote_to_anbu", {"ninja_name": "Sasuke Uchiha"}, call_id="c2")],
            ]
        )
        pending = await client.post("/chat", json={"ninja_id": "shikamaru-nara", "message": "Promote Sasuke Uchiha to ANBU."})
        print("[needs_approval]", pending.status_code, pending.json())

        # proves approving the saved RunState resumes the run to a final answer
        script_next_request([[assistant_message("Sasuke Uchiha is now ANBU. Congratulations!")]])
        resolved = await client.post(f"/approvals/{pending.json()['approval_id']}", json={"decision": "approve"})
        print("[resolved]", resolved.status_code, resolved.json())

        # proves /chat/stream sends SSE events and ends with a done event
        script_next_request([[assistant_message("Naruto Uzumaki is a Kage-level ninja.")]])
        async with client.stream(
            "POST", "/chat/stream", json={"ninja_id": "shikamaru-nara", "message": "Who is Naruto Uzumaki?"}
        ) as response:
            print("[stream]", response.status_code, response.headers.get("content-type"))
            async for line in response.aiter_lines():
                if line.strip():  # skip the blank lines that close each SSE event
                    print(" ", line)


await demo_app()
# Outside a notebook: asyncio.run(demo_app())

[blocked] 200 {'status': 'blocked', 'reason': 'forbidden jutsu detected'}
[needs_approval] 200 {'status': 'needs_approval', 'approval_id': '32ab2f0f-c825-475e-8a9c-8f294b7ef441', 'pending_tools': ['promote_to_anbu']}
[resolved] 200 {'status': 'complete', 'output': 'Sasuke Uchiha is now ANBU. Congratulations!', 'agent': 'ANBU Promotion Review'}
[stream] 200 text/event-stream; charset=utf-8
  data: {"type": "agent_updated_stream_event"}
  data: {"type": "raw_response_event"}
  data: {"type": "raw_response_event"}
  data: {"type": "raw_response_event"}
  data: {"type": "raw_response_event"}
  data: {"type": "raw_response_event"}
  data: {"type": "raw_response_event"}
  data: {"type": "raw_response_event"}
  data: {"type": "raw_response_event"}
  data: {"type": "raw_response_event"}
  data: {"type": "run_item_stream_event"}
  data: {"type": "done", "final_output": "Naruto Uzumaki is a Kage-level ninja."}


## Running it for real

With `OPENAI_API_KEY` set and no dependency override, this same app can serve real traffic:
```bash
uvicorn capstone:app --host 0.0.0.0 --port 8000
```
A small `Dockerfile` for it:
```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "capstone:app", "--host", "0.0.0.0", "--port", "8000"]
```

## Summary — and the series

- The Mission Ops Center runs as real FastAPI routes: checked by a guardrail, routed by a handoff, saved in sessions,
  held for human approval, and streamed over SSE. It is tested from start to finish with the HTTP layer left real.
  To serve real traffic, set `OPENAI_API_KEY`, leave the override alone, and run
  `uvicorn capstone:app --host 0.0.0.0 --port 8000`.
- The gotcha that closes the series: guardrails see the whole session history by default, and a blocked message
  is still saved. Check only the newest message, and call `session.pop_item()` after a block.

That's the whole series: setup through a production capstone, on the real installed
`openai-agents` SDK.